In [12]:
# Import required python modules
import os
import numpy as np
import matplotlib.pyplot as plt
import healpy as hp
import pandas as pd
import sqlite3
from astropy.coordinates import SkyCoord
import astropy.units as u
import rubin_scheduler.scheduler.utils as sched_utils
from rubin_scheduler.utils import ddf_locations, ddf_locations_skycoord
from rubin_sim.data import get_baseline
import rubin_sim.maf as maf
from scipy.stats import binned_statistic

from IPython.display import display, Markdown

In [13]:
# v2.2 for accordian series
basedir = '/sdf/group/rubin/web_data/sim-data/sims_featureScheduler_runs2.2/ddf_accourd'
ddf_accord = [run.replace(".db", "") for run in os.listdir(basedir) if "_10yrs.db" in run]
ddf_accord

['ddf_accourd_sf0.25_lsf0.4_lsr0.1_v2.2_10yrs',
 'ddf_accourd_sf0.10_lsf0.4_lsr0.1_v2.2_10yrs',
 'ddf_accourd_sf0.05_lsf0.3_lsr0.5_v2.2_10yrs',
 'ddf_accourd_sf0.25_lsf0.4_lsr0.5_v2.2_10yrs',
 'ddf_accourd_sf0.05_lsf0.4_lsr0.1_v2.2_10yrs',
 'ddf_accourd_sf0.20_lsf0.4_lsr0.1_v2.2_10yrs',
 'ddf_accourd_sf0.20_lsf0.4_lsr0.5_v2.2_10yrs',
 'ddf_accourd_sf0.05_lsf0.2_lsr0.5_v2.2_10yrs',
 'ddf_accourd_sf0.05_lsf0.1_lsr0.5_v2.2_10yrs',
 'ddf_accourd_sf0.05_lsf0.3_lsr0.1_v2.2_10yrs',
 'ddf_accourd_sf0.20_lsf0.3_lsr0.5_v2.2_10yrs',
 'ddf_accourd_sf0.05_lsf0.2_lsr0.1_v2.2_10yrs',
 'ddf_accourd_sf0.30_lsf0.4_lsr0.1_v2.2_10yrs',
 'ddf_accourd_sf0.10_lsf0.4_lsr0.3_v2.2_10yrs',
 'ddf_accourd_sf0.05_lsf0.2_lsr0.3_v2.2_10yrs',
 'ddf_accourd_sf0.05_lsf0.3_lsr0.3_v2.2_10yrs',
 'ddf_accourd_sf0.05_lsf0.1_lsr0.1_v2.2_10yrs',
 'ddf_accourd_sf0.25_lsf0.3_lsr0.5_v2.2_10yrs',
 'ddf_accourd_sf0.30_lsf0.4_lsr0.5_v2.2_10yrs',
 'ddf_accourd_sf0.05_lsf0.4_lsr0.3_v2.2_10yrs',
 'ddf_accourd_sf0.25_lsf0.3_lsr0.1_v2.2_

In [15]:
conn = sqlite3.connect(os.path.join(basedir, ddf_accord[0] + ".db"))
query = 'select night, moonPhase from observations where night<100'
vals = pd.read_sql(query, conn)
nstart = vals.query('moonPhase < 50').night.min()
print(nstart)

6


In [17]:
# Basic DDF information, number of visits from baseline simulation
ddfs = ddf_locations_skycoord()
nvis = {}
frac_seeing = {}
u_per_30 = {}
u_per_30 = {}
max_u_per_30 = {}
pair_count = {}
for run in ddf_accord:
    conn = sqlite3.connect(os.path.join(basedir, run + ".db"))
    nvis[run] = {}
    frac_seeing[run] = {}
    u_per_30[run] = {}
    max_u_per_30[run] = {}
    pair_count[run] = {}
    for i in ddfs:
        query = f"select count(*) from observations where note like '%{i}%'"
        nvis[run][i] = int(pd.read_sql(query, conn).values[0][0])
        query = f"select seeingFwhmEff from observations where note like '%{i}%' and filter == 'r'"
        seeing = pd.read_sql(query, conn)['seeingFwhmEff'].values
        frac_seeing[run][i] = len(np.where(seeing < 1.2)[0]) / len(seeing)
        query = f"select observationStartMJD, night, filter, moonPhase from observations where note like '%{i}%'"
        vals = pd.read_sql(query, conn)
        # Find times with 60 u band visits in 30 nights
        counts = vals.query("filter == 'u'").groupby('night').count()['observationStartMJD']
        visits_per_30, bin_edges, bin_count = binned_statistic(counts.index, counts.values, bins=np.arange(nstart, 3652, 30), statistic='sum')
        max_u_per_30[run][i] = max(visits_per_30)
        u_per_30[run][i] = len(np.where(visits_per_30 > 60)[0])
        # Find pairs of nights with the required number of visits per pair-night
        t = vals.groupby(['night', 'filter']).agg({'filter': 'count'}).rename(columns={'filter': 'count'}).reset_index().pivot(index=['night'], columns=['filter']).fillna(0)
        t = t.droplevel(level=0, axis=1)
        t = t.drop(labels='u', axis=1)
        t['total'] = t.sum(axis=1)
        t = t.query('total > 0')
        s = t.query('g>1 and r>1 and i>3 and z>5 and y>4')
        # find and drop all the impossible y combinations
        def find_drops(df, column, minimum_val):
            idx_bad = np.where(df[column] < minimum_val)[0]
            idx_good = np.where(df[column] >= minimum_val)[0]
            drop = []
            for i in idx_bad:
                if (i+1 not in idx_good) and (i-1 not in idx_good):
                    drop.append(i)
            return drop
        drop = find_drops(t, 'y', 4)
        t = t.drop(index=t.iloc[drop].index)
        drop = find_drops(t, 'z', 5)
        t = t.drop(index=t.iloc[drop].index)
        drop = find_drops(t, 'i', 3)
        t = t.drop(index=t.iloc[drop].index)
        drop = find_drops(t, 'r', 1)
        t = t.drop(index=t.iloc[drop].index)
        drop = find_drops(t, 'g', 1)
        t = t.drop(index=t.iloc[drop].index)
        nights = t.index.values
        pairs = np.where(np.diff(nights) < 2)[0]
        pnights = t.iloc[np.sort(np.concatenate([pairs, pairs+1]))].index.values
        double_count = [p for p in pnights if p in s.index.values]
        pair_count[run][i] = len(pairs) + len(s) - len(double_count)
        
nvisits = pd.DataFrame(nvis).T
seeing = pd.DataFrame(frac_seeing).T
max_u_per_30 = pd.DataFrame(max_u_per_30).T
u_per_30 = pd.DataFrame(u_per_30).T
night_pairs = pd.DataFrame(pair_count).T

In [18]:
display(Markdown("Number of visits per DDF"))
display(nvisits)

display(Markdown("Fraction of DDF r-band visits with seeing < 1.2\""))
display(seeing.round(2))

display(Markdown("Maximum number of u band visits within 30 nights"))
display(max_u_per_30)

display(Markdown("Number of months with >60 u band visits in 30 nights"))
display(u_per_30)

display(Markdown("Number of 48-hour intervals with g>1, r>1, i>3, z>5, y>4 visits"))
display(night_pairs)

Number of visits per DDF

,ELAISS1,XMM_LSS,ECDFS,COSMOS,EDFS_a,EDFS_b
ddf_accourd_sf0.25_lsf0.4_lsr0.1_v2.2_10yrs,19906,20980,21305,22718,10798,10776
ddf_accourd_sf0.10_lsf0.4_lsr0.1_v2.2_10yrs,19850,20470,21204,21271,10682,10644
ddf_accourd_sf0.05_lsf0.3_lsr0.5_v2.2_10yrs,19434,18518,19941,18338,10276,10184
ddf_accourd_sf0.25_lsf0.4_lsr0.5_v2.2_10yrs,20660,21381,20926,22262,10810,10780
ddf_accourd_sf0.05_lsf0.4_lsr0.1_v2.2_10yrs,19600,19797,20802,20706,10517,10496
ddf_accourd_sf0.20_lsf0.4_lsr0.1_v2.2_10yrs,20162,20608,21405,22830,10488,10465
ddf_accourd_sf0.20_lsf0.4_lsr0.5_v2.2_10yrs,20300,20333,20806,21243,10664,10644
ddf_accourd_sf0.05_lsf0.2_lsr0.5_v2.2_10yrs,19483,18648,20040,18701,10373,10324
ddf_accourd_sf0.05_lsf0.1_lsr0.5_v2.2_10yrs,19424,18305,19596,18088,10132,10066
ddf_accourd_sf0.05_lsf0.3_lsr0.1_v2.2_10yrs,20514,20563,21143,21769,10678,10641


Fraction of DDF r-band visits with seeing < 1.2"

,ELAISS1,XMM_LSS,ECDFS,COSMOS,EDFS_a,EDFS_b
ddf_accourd_sf0.25_lsf0.4_lsr0.1_v2.2_10yrs,0.68,0.69,0.80,0.71,0.80,0.82
ddf_accourd_sf0.10_lsf0.4_lsr0.1_v2.2_10yrs,0.67,0.67,0.75,0.69,0.76,0.76
ddf_accourd_sf0.05_lsf0.3_lsr0.5_v2.2_10yrs,0.60,0.61,0.65,0.64,0.68,0.67
ddf_accourd_sf0.25_lsf0.4_lsr0.5_v2.2_10yrs,0.66,0.70,0.77,0.73,0.78,0.77
ddf_accourd_sf0.05_lsf0.4_lsr0.1_v2.2_10yrs,0.63,0.65,0.74,0.67,0.72,0.74
ddf_accourd_sf0.20_lsf0.4_lsr0.1_v2.2_10yrs,0.71,0.68,0.80,0.71,0.80,0.79
ddf_accourd_sf0.20_lsf0.4_lsr0.5_v2.2_10yrs,0.71,0.63,0.74,0.67,0.77,0.73
ddf_accourd_sf0.05_lsf0.2_lsr0.5_v2.2_10yrs,0.63,0.59,0.65,0.65,0.62,0.62
ddf_accourd_sf0.05_lsf0.1_lsr0.5_v2.2_10yrs,0.61,0.58,0.60,0.65,0.59,0.66
ddf_accourd_sf0.05_lsf0.3_lsr0.1_v2.2_10yrs,0.65,0.67,0.74,0.72,0.71,0.69


Maximum number of u band visits within 30 nights

,ELAISS1,XMM_LSS,ECDFS,COSMOS,EDFS_a,EDFS_b
ddf_accourd_sf0.25_lsf0.4_lsr0.1_v2.2_10yrs,40.0,40.0,40.0,40.0,20.0,20.0
ddf_accourd_sf0.10_lsf0.4_lsr0.1_v2.2_10yrs,40.0,32.0,40.0,32.0,20.0,20.0
ddf_accourd_sf0.05_lsf0.3_lsr0.5_v2.2_10yrs,24.0,24.0,24.0,32.0,12.0,12.0
ddf_accourd_sf0.25_lsf0.4_lsr0.5_v2.2_10yrs,32.0,24.0,24.0,24.0,16.0,16.0
ddf_accourd_sf0.05_lsf0.4_lsr0.1_v2.2_10yrs,40.0,40.0,32.0,32.0,20.0,20.0
ddf_accourd_sf0.20_lsf0.4_lsr0.1_v2.2_10yrs,40.0,40.0,40.0,40.0,20.0,20.0
ddf_accourd_sf0.20_lsf0.4_lsr0.5_v2.2_10yrs,32.0,24.0,24.0,24.0,16.0,16.0
ddf_accourd_sf0.05_lsf0.2_lsr0.5_v2.2_10yrs,16.0,16.0,16.0,37.0,8.0,8.0
ddf_accourd_sf0.05_lsf0.1_lsr0.5_v2.2_10yrs,16.0,16.0,16.0,42.0,8.0,8.0
ddf_accourd_sf0.05_lsf0.3_lsr0.1_v2.2_10yrs,24.0,24.0,24.0,32.0,12.0,12.0


Number of months with >60 u band visits in 30 nights

,ELAISS1,XMM_LSS,ECDFS,COSMOS,EDFS_a,EDFS_b
ddf_accourd_sf0.25_lsf0.4_lsr0.1_v2.2_10yrs,0,0,0,0,0,0
ddf_accourd_sf0.10_lsf0.4_lsr0.1_v2.2_10yrs,0,0,0,0,0,0
ddf_accourd_sf0.05_lsf0.3_lsr0.5_v2.2_10yrs,0,0,0,0,0,0
ddf_accourd_sf0.25_lsf0.4_lsr0.5_v2.2_10yrs,0,0,0,0,0,0
ddf_accourd_sf0.05_lsf0.4_lsr0.1_v2.2_10yrs,0,0,0,0,0,0
ddf_accourd_sf0.20_lsf0.4_lsr0.1_v2.2_10yrs,0,0,0,0,0,0
ddf_accourd_sf0.20_lsf0.4_lsr0.5_v2.2_10yrs,0,0,0,0,0,0
ddf_accourd_sf0.05_lsf0.2_lsr0.5_v2.2_10yrs,0,0,0,0,0,0
ddf_accourd_sf0.05_lsf0.1_lsr0.5_v2.2_10yrs,0,0,0,0,0,0
ddf_accourd_sf0.05_lsf0.3_lsr0.1_v2.2_10yrs,0,0,0,0,0,0


Number of 48-hour intervals with g>1, r>1, i>3, z>5, y>4 visits

,ELAISS1,XMM_LSS,ECDFS,COSMOS,EDFS_a,EDFS_b
ddf_accourd_sf0.25_lsf0.4_lsr0.1_v2.2_10yrs,114,115,136,112,139,139
ddf_accourd_sf0.10_lsf0.4_lsr0.1_v2.2_10yrs,119,124,141,105,138,139
ddf_accourd_sf0.05_lsf0.3_lsr0.5_v2.2_10yrs,128,127,130,110,140,139
ddf_accourd_sf0.25_lsf0.4_lsr0.5_v2.2_10yrs,143,143,141,139,149,148
ddf_accourd_sf0.05_lsf0.4_lsr0.1_v2.2_10yrs,119,117,133,100,138,138
ddf_accourd_sf0.20_lsf0.4_lsr0.1_v2.2_10yrs,125,117,136,116,132,131
ddf_accourd_sf0.20_lsf0.4_lsr0.5_v2.2_10yrs,138,141,142,123,152,152
ddf_accourd_sf0.05_lsf0.2_lsr0.5_v2.2_10yrs,124,127,139,115,143,144
ddf_accourd_sf0.05_lsf0.1_lsr0.5_v2.2_10yrs,123,122,129,104,136,135
ddf_accourd_sf0.05_lsf0.3_lsr0.1_v2.2_10yrs,131,128,142,135,144,143
